# Chess-Coach Agent — E4B QLoRA on **Google Colab** (single T4, de-risk)

Probe E4B fit at seq 1152 on one T4, then a short end-to-end train. Mirrors `kaggle_e4b_qlora.ipynb` so moving to the Kaggle 30h slot is near-copy: same cells, same flags — only paths/Secrets + MAX_STEPS differ. If the probe (Cell 6.5) reports OOM, lower MAX_SEQ to 1024 in Cell 1 and re-run.


In [ ]:
# Cell 1 — config (E4B QLoRA on ONE T4; mirrors kaggle_e4b for easy migration)
import os

MODEL = "gemma4_e4b"  # E4B in 4-bit on ONE T4 (single-GPU; DDP is what OOMd, not size)
HF_REPO = {
    "gemma4_e4b": "google/gemma-4-E4B-it-qat-q4_0-unquantized",
    "gemma4_e2b": "google/gemma-4-E2B-it-qat-q4_0-unquantized",
}[MODEL]
NO_4BIT = False  # 4-bit QLoRA: E4B must be 4-bit to fit 16 GB (bf16 E4B will not)

CHESS_REPO_URL = "https://github.com/RyanDev1st/llm-and-engine.git"
CHESS_BRANCH = "feat/chess-coach-sft"
WORKDIR = "/content/llm-and-engine"
OUTPUT = "gemma4_chess_e4b_colab"

USE_DRIVE = True
DRIVE_DIR = "/content/drive/MyDrive/chess_coach_runs"

# Colab = DE-RISK run (probe fit + confirm the E4B loop runs end-to-end). The full
# 30h train is reserved for Kaggle. seq 1152 is the test ceiling; if the probe OOMs,
# drop MAX_SEQ to 1024. E4B ~2x E2B compute, so keep MAX_STEPS small here.
MAX_SEQ = 1152
TARGETS = "attn-only"   # memory-safe on a 16 GB T4 (all-linear is heavier)
RANK = 16
GRAD_ACCUM = 8
MAX_STEPS = 120         # short de-risk; raise on Kaggle 30h slot
EVAL_EVERY = 40
MAX_VAL = 128
print("config:", MODEL, "seq=", MAX_SEQ, "steps=", MAX_STEPS, "drive=", USE_DRIVE)


In [ ]:
# Cell 2 — GPU preflight + wall-time projection
import subprocess, torch
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.free",
                      "--format=csv"], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), "No GPU — Runtime > Change runtime type > T4 GPU"
name = torch.cuda.get_device_name(0)
per = {"A100": 14, "L4": 32, "T4": 65}
sec = next((v for k, v in per.items() if k in name), 65)
print(f"device={name} | ~{sec}s/update -> {MAX_STEPS} steps ~= {MAX_STEPS*sec/3600:.1f}h "
      f"(raise MAX_STEPS if this is well under your session limit)")

In [ ]:
# Cell 3 — clone repo (code + data; skip LFS weights)
import os, subprocess

def run(cmd, **kw):
    print(">", " ".join(cmd)); return subprocess.run(cmd, check=True, text=True, **kw)

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
if not os.path.exists(WORKDIR):
    run(["git", "clone", "--depth", "1", "--branch", CHESS_BRANCH, CHESS_REPO_URL, WORKDIR], env=env)
else:
    run(["git", "-C", WORKDIR, "pull", "--ff-only"], env=env)
run(["git", "-C", WORKDIR, "log", "-1", "--oneline"])
os.environ["PYTHONPATH"] = f"{WORKDIR}/src/llm"
print("PYTHONPATH=", os.environ["PYTHONPATH"])

In [ ]:
# Cell 4 — deps (Gemma 4 needs recent transformers; drop torchao like Kaggle)
import subprocess, sys
def pip(*a):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a], check=True)
pip("-U", "transformers>=5.8", "accelerate>=1.2", "peft>=0.14", "trl>=0.13",
    "bitsandbytes>=0.45", "datasets", "sentencepiece", "protobuf", "python-chess")
# Colab may ship torchao; recent PEFT raises on old versions inside get_peft_model.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"])
import transformers, peft, bitsandbytes
print("transformers", transformers.__version__, "| peft", peft.__version__,
      "| bnb", bitsandbytes.__version__)

In [ ]:
# Cell 5 — HF login (Colab Secrets) + download base model
import os
from huggingface_hub import login, snapshot_download
try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = os.environ.get("HF_TOKEN")
assert token, "Add HF_TOKEN in the Colab Secrets panel (key icon) and enable notebook access."
login(token)
dest = f"{WORKDIR}/src/llm/models/{MODEL}"
snapshot_download(repo_id=HF_REPO, local_dir=dest,
                  allow_patterns=["*.json", "*.safetensors", "*.model", "*.txt", "tokenizer*"])
print("base model at", dest)
print(sorted(os.listdir(dest)))

In [ ]:
# Cell 6 — data check (REGENERATED qualitative corpus; stored gzipped)
import os, gzip
for name in ("v1_2_train.jsonl", "v1_2_val.jsonl"):
    base = f"{WORKDIR}/data/sft/{name}"
    path = base if os.path.exists(base) else base + ".gz"
    if not os.path.exists(path):
        n = 0
    elif path.endswith(".gz"):
        with gzip.open(path, "rt", encoding="utf-8") as fh:
            n = sum(1 for _ in fh)
    else:
        n = sum(1 for _ in open(path, encoding="utf-8"))
    print(name, "rows=", n, "(", os.path.basename(path), ")")
    assert n > 0, f"missing/empty {path} - pull the branch again"

In [ ]:
# Cell 6.5 — MEMORY PROBE: does E4B QLoRA fit ONE T4 at MAX_SEQ? Run BEFORE training.
import subprocess, sys, os
base = f"{WORKDIR}/src/llm/models/{MODEL}"
print(f"probing E4B fit at seq={MAX_SEQ} targets={TARGETS} ...")
subprocess.run([sys.executable, "-m", "llm_training.e4b_probe"],
               check=True, cwd=WORKDIR,
               env={**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm",
                    "CHESS_BASE": base, "PROBE_SEQ": str(MAX_SEQ),
                    "PROBE_TARGETS": TARGETS, "PROBE_RANK": str(RANK)})
# FIT/TIGHT -> run Cell 7. OOM -> lower MAX_SEQ in Cell 1 (try 1024) and re-run this cell.


In [ ]:
# Cell 7 — (optional) Drive for checkpoint persistence, then train. Drive is NON-FATAL:
# if mount fails (auth popup blocked/transient), fall back to local runs/ and keep going.
import subprocess, sys, os
mounted = False
if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
        runs = f"{WORKDIR}/runs"
        if not os.path.islink(runs) and not os.path.exists(runs):
            os.symlink(DRIVE_DIR, runs)
        mounted = True
        print("checkpoints ->", DRIVE_DIR)
    except Exception as e:
        print(f"Drive mount failed ({e}); using LOCAL runs/ (download via Cell 8). "
              "To use Drive: re-run this cell and complete the auth popup, or set USE_DRIVE=False.")
if not mounted:
    os.makedirs(f"{WORKDIR}/runs", exist_ok=True)

cmd = [sys.executable, "-m", "llm_training.run_train",
       "--model", MODEL, "--max-steps", str(MAX_STEPS), "--rank", str(RANK),
       "--targets", TARGETS, "--grad-accum", str(GRAD_ACCUM), "--max-seq", str(MAX_SEQ),
       "--eval-every", str(EVAL_EVERY), "--max-val", str(MAX_VAL),
       "--output", OUTPUT]
if NO_4BIT:
    cmd.append("--no-4bit")
print(">", " ".join(cmd))
subprocess.run(cmd, check=True, cwd=WORKDIR,
               env={**os.environ, "PYTHONPATH": f"{WORKDIR}/src/llm",
                    "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})


In [ ]:
# Cell 8 — zip the adapter + download (also already on Drive if USE_DRIVE)
import shutil, os
run_dir = f"{WORKDIR}/runs/{OUTPUT}"
print("adapter files:", os.listdir(run_dir))
out_zip = f"/content/{OUTPUT}_adapter"
shutil.make_archive(out_zip, "zip", run_dir)
print("zip at", out_zip + ".zip")
try:
    from google.colab import files
    files.download(out_zip + ".zip")
except Exception as e:
    print("download from the Files panel instead:", out_zip + ".zip", "|", e)